In [1]:
import torch
print(torch.__version__, torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("No GPU: Runtime -> Change runtime type -> T4 GPU")
print(torch.cuda.get_device_name(0))

2.11.0+cu128 True
Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:

ZIP = '/content/drive/MyDrive/submission.zip'

!rm -rf /content/submission
!unzip -q "$ZIP" -d /content/
%cd /content/submission
!ls

/content/submission
check_leakage.py	export_embeddings.py	model.py
colab_train.ipynb	gallery_canvas.py	README.md
dataset			gallery_probe.py	requirements.txt
dataset_preparation.py	generate_pairs.py	roc_analysis.py
dataset_stats.json	identity_audit.py	splits.json
demo_api.py		identity_manifest.json	train.py
error_analysis.py	landmarks.json
evaluate_pairs.py	make_summary.py


In [5]:
import json
splits = json.load(open('splits.json'))
for k, v in splits.items():
    print(k, len(v), 'identities')

overlap = set(splits['train']) & (set(splits['val']) | set(splits['test']))
print('train/eval overlap:', len(overlap))
assert not overlap

train 1500 identities
val 60 identities
test 120 identities
train/eval overlap: 0


In [6]:
!python train.py --root . --size 224 --epochs 30 \
    --batch-p 32 --batch-k 4 --lr 1e-3 --workers 2

device=cuda
7096 images / 1500 identities, 55 batches of 128
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100% 97.8M/97.8M [00:00<00:00, 180MB/s]
[  1/30] loss=20.6724 (arc=20.1742 tri=0.4983) acc=0.001 | val AUC=0.7887 EER=0.2808 | 42s
        saved (val AUC 0.7887)
[  2/30] loss=18.9577 (arc=18.5339 tri=0.4238) acc=0.005 | val AUC=0.8201 EER=0.2555 | 40s
        saved (val AUC 0.8201)
[  3/30] loss=17.7693 (arc=17.3766 tri=0.3927) acc=0.005 | val AUC=0.8212 EER=0.2548 | 42s
        saved (val AUC 0.8212)
[  4/30] loss=15.6375 (arc=15.2917 tri=0.3458) acc=0.013 | val AUC=0.8789 EER=0.2060 | 42s
        saved (val AUC 0.8789)
[  5/30] loss=13.4810 (arc=13.1850 tri=0.2960) acc=0.029 | val AUC=0.8929 EER=0.1852 | 41s
        saved (val AUC 0.8929)
[  6/30] loss=11.5615 (arc=11.3064 tri=0.2551) acc=0.055 | val AUC=0.8995 EER=0.1800 | 41s
        saved (val AUC 0.8995)
[  7/30] loss=9.9292 (arc=9.7045 t

In [7]:
import json
h = json.load(open('checkpoints/training_history.json'))
for r in h['history']:
    print(f"epoch {r['epoch']:3d}  loss {r['loss']:.4f}  acc {r['train_acc']:.3f}  "
          f"val AUC {r['val_auc']:.4f}  val EER {r['val_eer']:.4f}")
print('best val AUC:', h['best_val_auc'])

epoch   1  loss 20.6724  acc 0.001  val AUC 0.7887  val EER 0.2808
epoch   2  loss 18.9577  acc 0.005  val AUC 0.8201  val EER 0.2555
epoch   3  loss 17.7693  acc 0.005  val AUC 0.8212  val EER 0.2548
epoch   4  loss 15.6375  acc 0.013  val AUC 0.8789  val EER 0.2060
epoch   5  loss 13.4810  acc 0.029  val AUC 0.8929  val EER 0.1852
epoch   6  loss 11.5615  acc 0.055  val AUC 0.8995  val EER 0.1800
epoch   7  loss 9.9292  acc 0.097  val AUC 0.9200  val EER 0.1545
epoch   8  loss 8.5625  acc 0.153  val AUC 0.9178  val EER 0.1595
epoch   9  loss 7.4112  acc 0.214  val AUC 0.9171  val EER 0.1588
epoch  10  loss 6.5800  acc 0.272  val AUC 0.9196  val EER 0.1507
epoch  11  loss 5.9058  acc 0.335  val AUC 0.9319  val EER 0.1412
epoch  12  loss 5.2604  acc 0.396  val AUC 0.9246  val EER 0.1460
epoch  13  loss 4.6849  acc 0.466  val AUC 0.9320  val EER 0.1330
epoch  14  loss 4.2859  acc 0.520  val AUC 0.9381  val EER 0.1352
epoch  15  loss 3.9042  acc 0.563  val AUC 0.9368  val EER 0.1327
epoc

In [8]:
!cp checkpoints/best_model.pth checkpoints/training_history.json /content/drive/MyDrive/

from google.colab import files
files.download('checkpoints/best_model.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>